# 03 — Strategy Analysis

Formula 1 race strategy — which tyre you use and when you pit — is often the difference  
between winning and finishing outside the points.

| Section | Focus |
|---------|-------|
| 1 | Tyre compound usage across the season |
| 2 | Stint length distributions |
| 3 | Tyre degradation (pace drop per lap) |
| 4 | Pit stop timing & undercut windows |
| 5 | Strategy chains per race |

> **Prerequisite:** Run `01_data_collection.ipynb` first.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, os.path.join('..', 'src'))
from data_processing import load_lap_data, summarize_strategy

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

COMPOUND_COLORS = {
    'SOFT':         '#FF3333',
    'MEDIUM':       '#FFF200',
    'HARD':         '#EBEBEB',
    'INTERMEDIATE': '#43B02A',
    'WET':          '#0067FF',
    'UNKNOWN':      '#999999',
}

In [ ]:
RAW_PATH = os.path.join('..', 'data', 'raw', 'all_grand_prix_laps_2025.csv')
df = load_lap_data(RAW_PATH)

# Valid laps with compound data
laps = df[
    df['LapTime'].notna() &
    df['Compound'].notna() &
    df['Deleted'].eq(False)
].copy()

laps['Compound'] = laps['Compound'].str.upper().str.strip()
laps['is_pit_lap'] = laps['PitInTime'].notna() | laps['PitOutTime'].notna()

print(f'Laps with compound data: {len(laps):,}')
print(f'Compounds seen: {sorted(laps["Compound"].unique())}')

## 1 · Tyre Compound Usage — Season Overview

In [ ]:
compound_counts = laps['Compound'].value_counts().reset_index()
compound_counts.columns = ['Compound', 'laps']

fig = px.pie(
    compound_counts,
    names='Compound',
    values='laps',
    color='Compound',
    color_discrete_map=COMPOUND_COLORS,
    title='Tyre Compound Usage — 2025 Season (all laps)',
    hole=0.35,
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Compound mix per race (dry compounds only)
dry = laps[laps['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])].copy()

mix = (
    dry.groupby(['EventName', 'RoundNumber', 'Compound'])
    .size()
    .reset_index(name='laps')
    .sort_values('RoundNumber')
)

fig = px.bar(
    mix,
    x='EventName',
    y='laps',
    color='Compound',
    color_discrete_map=COMPOUND_COLORS,
    barmode='stack',
    title='Dry Compound Usage per Race — 2025 Season',
    labels={'laps': 'Laps', 'EventName': ''},
    height=480,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## 2 · Stint Length Distributions

How many laps do drivers typically stay out on each compound?

In [ ]:
# Stint length = max TyreLife seen per driver/event/stint
stint_lengths = (
    laps.groupby(['EventName', 'Driver', 'Stint'])['TyreLife']
    .max()
    .reset_index(name='stint_laps')
    .merge(
        laps[['EventName','Driver','Stint','Compound']].drop_duplicates(),
        on=['EventName','Driver','Stint'],
        how='left'
    )
)
stint_lengths = stint_lengths[stint_lengths['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])]

fig = px.box(
    stint_lengths,
    x='Compound',
    y='stint_laps',
    color='Compound',
    color_discrete_map=COMPOUND_COLORS,
    category_orders={'Compound': ['SOFT', 'MEDIUM', 'HARD']},
    title='Stint Length Distribution by Compound — 2025 Season',
    labels={'stint_laps': 'Laps in Stint', 'Compound': ''},
    points='all',
    height=450,
)
fig.update_layout(showlegend=False)
fig.show()

## 3 · Tyre Degradation

We measure degradation as the rate at which lap time increases with tyre age (laps on tyre).  
Using linear regression per compound gives a degradation slope in seconds/lap.

In [ ]:
from scipy import stats as sp_stats

deg_data = dry[~dry['is_pit_lap'] & dry['IsAccurate'].eq(True)].copy()

# Average lap time per tyre age + compound across the season
deg_avg = (
    deg_data.groupby(['Compound', 'TyreLife'])['LapTime']
    .median()
    .reset_index()
)

fig = px.scatter(
    deg_avg,
    x='TyreLife',
    y='LapTime',
    color='Compound',
    color_discrete_map=COMPOUND_COLORS,
    trendline='ols',
    title='Tyre Degradation — Median Lap Time vs Tyre Age (2025 Season)',
    labels={'TyreLife': 'Tyre Age (laps)', 'LapTime': 'Median Lap Time (s)'},
    height=480,
)
fig.show()

# Print degradation slope per compound
print('Degradation slope (s per lap):')
for compound in ['SOFT', 'MEDIUM', 'HARD']:
    sub = deg_avg[deg_avg['Compound'] == compound].dropna()
    if len(sub) >= 3:
        slope, intercept, r, p, se = sp_stats.linregress(sub['TyreLife'], sub['LapTime'])
        print(f'  {compound:10s}: {slope:+.4f} s/lap  (R²={r**2:.3f})')

## 4 · Pit Stop Timing

When drivers pit (as a lap number) — early pitters vs late pitters, and how lap traffic plays a role.

In [ ]:
# Pit-in laps
pit_ins = laps[laps['PitInTime'].notna()][['EventName', 'RoundNumber', 'Driver', 'Team', 'LapNumber']].copy()
pit_ins = pit_ins.sort_values('RoundNumber')

# Total laps per race for normalisation
race_lengths = (
    laps.groupby('EventName')['LapNumber']
    .max()
    .rename('total_laps')
    .reset_index()
)
pit_ins = pit_ins.merge(race_lengths, on='EventName')
pit_ins['pit_pct'] = pit_ins['LapNumber'] / pit_ins['total_laps'] * 100

fig = px.histogram(
    pit_ins,
    x='pit_pct',
    nbins=40,
    title='Pit Stop Timing Distribution (% of race distance)',
    labels={'pit_pct': '% of race completed at pit', 'count': 'Frequency'},
    color_discrete_sequence=['#FF8000'],
    height=420,
)
fig.update_layout(bargap=0.05)
fig.show()

## 5 · Strategy Chains per Race

What compound sequences were used and which led to better finishing positions?

In [ ]:
# Build strategy string per driver per race
strategy = (
    laps.groupby(['EventName', 'RoundNumber', 'Driver', 'Team'])
    .apply(lambda g: summarize_strategy(g.sort_values('LapNumber')['Compound']))
    .reset_index(name='strategy')
)

# Final position per driver per race
final_pos = (
    df.sort_values(['EventName', 'Driver', 'LapNumber'])
    .groupby(['EventName', 'Driver'])['Position']
    .last()
    .reset_index(name='final_position')
)

strat_perf = strategy.merge(final_pos, on=['EventName', 'Driver'], how='left')
strat_perf['final_position'] = pd.to_numeric(strat_perf['final_position'], errors='coerce')

# Most common strategies
top_strategies = (
    strat_perf['strategy']
    .value_counts()
    .head(15)
    .reset_index()
)
top_strategies.columns = ['strategy', 'count']

fig = px.bar(
    top_strategies,
    x='count',
    y='strategy',
    orientation='h',
    title='Top 15 Race Strategies — 2025 Season',
    labels={'count': 'Times Used', 'strategy': 'Strategy'},
    color='count',
    color_continuous_scale='Oranges',
    height=520,
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.show()

In [ ]:
# Average finishing position per strategy (top strategies only)
top_strat_names = top_strategies['strategy'].tolist()
strat_result = (
    strat_perf[strat_perf['strategy'].isin(top_strat_names)]
    .groupby('strategy')['final_position']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_finish', 'count': 'uses'})
    .sort_values('avg_finish')
)

fig = px.bar(
    strat_result,
    x='avg_finish',
    y='strategy',
    orientation='h',
    title='Average Finishing Position by Strategy (lower = better)',
    labels={'avg_finish': 'Avg Finishing Position', 'strategy': 'Strategy'},
    color='avg_finish',
    color_continuous_scale='RdYlGn_r',
    height=480,
    text='avg_finish',
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total descending'}, coloraxis_showscale=False)
fig.show()

---
## ✅ Key Takeaways

Fill these in after running:

- **Most used strategy:** ...
- **Best performing strategy (avg finish):** ...
- **Fastest degrading compound:** ...
- **Typical pit window:** ...

Proceed to **[04_feature_engineering.ipynb](04_feature_engineering.ipynb)**.